# GridWise Notebook 1: Karnataka Synthetic Energy Data (2019-2024)
Generates hourly district-city synthetic grid data and exports district-year CSV files plus one combined file.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', 'numpy', 'holidays', 'faker', 'joblib', '-q'], check=True)
import pandas as pd
import numpy as np
import holidays
from datetime import datetime
import os

from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/gridwise_data/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

rng = np.random.default_rng(42)
ka_holidays = holidays.India(state='KA')

In [ ]:
KARNATAKA_GRID = {
    'Bangalore Urban': {'cities': {'Bengaluru': {'household_count': 500, 'base_demand_kWh': 3.8, 'prosumer_base_pct': 0.12, 'zone': 'urban'}, 'Whitefield': {'household_count': 300, 'base_demand_kWh': 3.2, 'prosumer_base_pct': 0.18, 'zone': 'urban'}, 'Electronic City': {'household_count': 250, 'base_demand_kWh': 2.9, 'prosumer_base_pct': 0.15, 'zone': 'urban'}}},
    'Bangalore Rural': {'cities': {'Devanahalli': {'household_count': 120, 'base_demand_kWh': 2.3, 'prosumer_base_pct': 0.14, 'zone': 'semi-urban'}, 'Doddaballapur': {'household_count': 110, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.16, 'zone': 'semi-urban'}, 'Nelamangala': {'household_count': 100, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.13, 'zone': 'semi-urban'}}},
    'Mysuru': {'cities': {'Mysuru': {'household_count': 260, 'base_demand_kWh': 2.8, 'prosumer_base_pct': 0.11, 'zone': 'urban'}, 'Nanjangud': {'household_count': 120, 'base_demand_kWh': 2.2, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Hunsur': {'household_count': 90, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.09, 'zone': 'semi-urban'}}},
    'Tumkur': {'cities': {'Tumkur': {'household_count': 180, 'base_demand_kWh': 2.5, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Tiptur': {'household_count': 90, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.11, 'zone': 'rural'}, 'Madhugiri': {'household_count': 80, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Mandya': {'cities': {'Mandya': {'household_count': 140, 'base_demand_kWh': 2.3, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Maddur': {'household_count': 85, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.11, 'zone': 'rural'}, 'Srirangapatna': {'household_count': 75, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.10, 'zone': 'rural'}}},
    'Hassan': {'cities': {'Hassan': {'household_count': 130, 'base_demand_kWh': 2.2, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Belur': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'rural'}, 'Sakleshpur': {'household_count': 65, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.13, 'zone': 'hill'}}},
    'Kodagu': {'cities': {'Madikeri': {'household_count': 60, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.05, 'zone': 'hill'}, 'Somwarpet': {'household_count': 50, 'base_demand_kWh': 1.6, 'prosumer_base_pct': 0.06, 'zone': 'hill'}, 'Kushalnagar': {'household_count': 55, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.07, 'zone': 'hill'}}},
    'Chikkamagaluru': {'cities': {'Chikkamagaluru': {'household_count': 70, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.07, 'zone': 'hill'}, 'Mudigere': {'household_count': 45, 'base_demand_kWh': 1.6, 'prosumer_base_pct': 0.08, 'zone': 'hill'}, 'Koppa': {'household_count': 40, 'base_demand_kWh': 1.5, 'prosumer_base_pct': 0.08, 'zone': 'hill'}}},
    'Shivamogga': {'cities': {'Shivamogga': {'household_count': 150, 'base_demand_kWh': 2.4, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Sagar': {'household_count': 80, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.11, 'zone': 'hill'}, 'Bhadravathi': {'household_count': 95, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}}},
    'Davanagere': {'cities': {'Davanagere': {'household_count': 160, 'base_demand_kWh': 2.5, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Harihar': {'household_count': 90, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Jagaluru': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Chitradurga': {'cities': {'Chitradurga': {'household_count': 120, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Hosadurga': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'rural'}, 'Challakere': {'household_count': 75, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.13, 'zone': 'rural'}}},
    'Bellary': {'cities': {'Ballari': {'household_count': 170, 'base_demand_kWh': 2.6, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Hospet': {'household_count': 110, 'base_demand_kWh': 2.2, 'prosumer_base_pct': 0.13, 'zone': 'semi-urban'}, 'Sandur': {'household_count': 65, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.14, 'zone': 'rural'}}},
    'Raichur': {'cities': {'Raichur': {'household_count': 140, 'base_demand_kWh': 2.5, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Sindhanur': {'household_count': 90, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.12, 'zone': 'rural'}, 'Manvi': {'household_count': 65, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Koppal': {'cities': {'Koppal': {'household_count': 110, 'base_demand_kWh': 2.2, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Gangavathi': {'household_count': 85, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Kushtagi': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.13, 'zone': 'rural'}}},
    'Gadag': {'cities': {'Gadag': {'household_count': 100, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Betageri': {'household_count': 75, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Ron': {'household_count': 55, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.13, 'zone': 'rural'}}},
    'Dharwad': {'cities': {'Dharwad': {'household_count': 140, 'base_demand_kWh': 2.4, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Hubballi': {'household_count': 220, 'base_demand_kWh': 2.8, 'prosumer_base_pct': 0.13, 'zone': 'urban'}, 'Alnavar': {'household_count': 60, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.11, 'zone': 'rural'}}},
    'Belgaum': {'cities': {'Belagavi': {'household_count': 210, 'base_demand_kWh': 2.7, 'prosumer_base_pct': 0.12, 'zone': 'urban'}, 'Gokak': {'household_count': 95, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.13, 'zone': 'semi-urban'}, 'Bailhongal': {'household_count': 65, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Haveri': {'cities': {'Haveri': {'household_count': 90, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Ranebennur': {'household_count': 95, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Byadagi': {'household_count': 50, 'base_demand_kWh': 1.6, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Uttara Kannada': {'cities': {'Karwar': {'household_count': 85, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.14, 'zone': 'coastal'}, 'Sirsi': {'household_count': 65, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.13, 'zone': 'hill'}, 'Kumta': {'household_count': 55, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.14, 'zone': 'coastal'}}},
    'Udupi': {'cities': {'Udupi': {'household_count': 95, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.15, 'zone': 'coastal'}, 'Manipal': {'household_count': 70, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.14, 'zone': 'coastal'}, 'Kundapur': {'household_count': 60, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.15, 'zone': 'coastal'}}},
    'Dakshina Kannada': {'cities': {'Mangaluru': {'household_count': 220, 'base_demand_kWh': 2.9, 'prosumer_base_pct': 0.14, 'zone': 'coastal'}, 'Puttur': {'household_count': 85, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.13, 'zone': 'coastal'}, 'Bantwal': {'household_count': 80, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.13, 'zone': 'coastal'}}},
    'Chikkaballapur': {'cities': {'Chikkaballapur': {'household_count': 95, 'base_demand_kWh': 2.0, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Kolar Gold Fields': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Gauribidanur': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Kolar': {'cities': {'Kolar': {'household_count': 100, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Malur': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'rural'}, 'Bangarpet': {'household_count': 65, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Ramanagara': {'cities': {'Ramanagara': {'household_count': 85, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Channapatna': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Magadi': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.13, 'zone': 'rural'}}},
    'Chamarajanagar': {'cities': {'Chamarajanagar': {'household_count': 75, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.10, 'zone': 'rural'}, 'Kollegal': {'household_count': 65, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.11, 'zone': 'rural'}, 'Gundlupete': {'household_count': 55, 'base_demand_kWh': 1.6, 'prosumer_base_pct': 0.11, 'zone': 'rural'}}},
    'Bidar': {'cities': {'Bidar': {'household_count': 95, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Basavakalyan': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.11, 'zone': 'semi-urban'}, 'Bhalki': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.11, 'zone': 'rural'}}},
    'Kalaburagi': {'cities': {'Kalaburagi': {'household_count': 150, 'base_demand_kWh': 2.5, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Sedam': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.11, 'zone': 'rural'}, 'Afzalpur': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.11, 'zone': 'rural'}}},
    'Yadgir': {'cities': {'Yadgir': {'household_count': 85, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Shorapur': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.11, 'zone': 'rural'}, 'Gurmitkal': {'household_count': 55, 'base_demand_kWh': 1.6, 'prosumer_base_pct': 0.11, 'zone': 'rural'}}},
    'Vijayapura': {'cities': {'Vijayapura': {'household_count': 125, 'base_demand_kWh': 2.3, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Sindagi': {'household_count': 70, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.11, 'zone': 'rural'}, 'Muddebihal': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}},
    'Bagalkot': {'cities': {'Bagalkot': {'household_count': 110, 'base_demand_kWh': 2.2, 'prosumer_base_pct': 0.10, 'zone': 'semi-urban'}, 'Badami': {'household_count': 65, 'base_demand_kWh': 1.8, 'prosumer_base_pct': 0.11, 'zone': 'rural'}, 'Jamkhandi': {'household_count': 70, 'base_demand_kWh': 1.9, 'prosumer_base_pct': 0.11, 'zone': 'rural'}}},
    'Vijayanagara': {'cities': {'Hosapete': {'household_count': 90, 'base_demand_kWh': 2.1, 'prosumer_base_pct': 0.12, 'zone': 'semi-urban'}, 'Harapanahalli': {'household_count': 60, 'base_demand_kWh': 1.7, 'prosumer_base_pct': 0.12, 'zone': 'rural'}, 'Hagaribommanahalli': {'household_count': 50, 'base_demand_kWh': 1.6, 'prosumer_base_pct': 0.12, 'zone': 'rural'}}}
}

COASTAL = {'Udupi', 'Dakshina Kannada', 'Uttara Kannada'}
HILL = {'Kodagu', 'Chikkamagaluru', 'Hassan', 'Shivamogga'}
NORTHERN_DRY = {'Raichur', 'Vijayapura', 'Yadgir', 'Kalaburagi', 'Bagalkot'}

In [ ]:
def season_from_month(month):
    if month in [3, 4, 5]:
        return 'Summer'
    if month in [6, 7, 8, 9]:
        return 'SW_Monsoon'
    if month in [10, 11]:
        return 'NE_Monsoon'
    return 'Winter'

def normalize_district_name(name):
    return name.lower().replace(' ', '_')

def build_city_frame(district, city, city_cfg, year, timestamps, city_index):
    n = len(timestamps)
    hour = timestamps.hour.values
    month = timestamps.month.values
    dow = timestamps.dayofweek.values

    growth_factor = (1.05) ** (year - 2019)
    adjusted_base_demand = city_cfg['base_demand_kWh'] * growth_factor
    adjusted_prosumer_pct = min(city_cfg['prosumer_base_pct'] * growth_factor, 0.60)

    is_weekend = np.isin(dow, [5, 6])
    is_holiday = pd.Series(timestamps.date).isin(ka_holidays).values
    season = np.array([season_from_month(m) for m in month])

    zone = city_cfg['zone']

    if zone == 'coastal':
        temp_base = rng.uniform(26, 33, n)
        humidity = rng.uniform(70, 95, n)
    elif zone == 'hill':
        temp_base = rng.uniform(14, 26, n)
        humidity = rng.uniform(55, 90, n)
    elif district in NORTHERN_DRY:
        temp_base = rng.uniform(30, 44, n)
        humidity = rng.uniform(30, 60, n)
    else:
        temp_base = rng.uniform(24, 38, n)
        humidity = rng.uniform(35, 75, n)

    temp_adj = np.where(season == 'Summer', 2.5, np.where(season == 'Winter', -3.0, -0.8))
    temperature = np.clip(temp_base + temp_adj, 12, 46)

    monsoon_mask = np.isin(month, [6, 7, 8, 9, 10, 11])
    cloudy_prob = np.where(monsoon_mask, 0.65, 0.25)
    rainy_prob = np.where(monsoon_mask, 0.45, 0.08)

    if district in COASTAL or zone == 'coastal':
        rainy_prob += 0.15
        cloudy_prob += 0.10
    if district in HILL or zone == 'hill':
        rainy_prob += 0.10

    is_cloudy = rng.random(n) < np.clip(cloudy_prob, 0.05, 0.95)
    is_rainy = rng.random(n) < np.clip(rainy_prob, 0.02, 0.90)

    day_mask = (hour >= 6) & (hour <= 18)
    solar_shape = np.zeros(n)
    solar_shape[day_mask] = np.sin(((hour[day_mask] - 6) / 12) * np.pi)

    max_irr = np.where(district in NORTHERN_DRY, 1100, np.where(zone == 'coastal', 900, 980))
    solar_irradiance = solar_shape * max_irr
    solar_irradiance = np.where(is_cloudy, solar_irradiance * 0.75, solar_irradiance)
    solar_irradiance = np.where(is_rainy, solar_irradiance * 0.55, solar_irradiance)
    solar_irradiance = np.where(day_mask, solar_irradiance, 0)

    wind_speed = np.where(zone == 'coastal', rng.uniform(15, 35, n), rng.uniform(5, 18, n))

    peak_flag = np.isin(hour, [6, 7, 8, 19, 20, 21, 22]).astype(int)

    weather_adjustment_factor = np.ones(n)
    weather_adjustment_factor += np.where(temperature > 35, 0.15, 0.0)
    weather_adjustment_factor += np.where(temperature > 40, 0.25, 0.0)
    weather_adjustment_factor += np.where(is_rainy, -0.05, 0.0)
    weather_adjustment_factor += np.where(is_holiday, -0.10, 0.0)
    weather_adjustment_factor += np.where(peak_flag == 1, 0.20, 0.0)
    weather_adjustment_factor = np.clip(weather_adjustment_factor, 0.45, None)

    adjusted_demand = adjusted_base_demand * city_cfg['household_count'] * weather_adjustment_factor

    panel_efficiency = 0.18 - (year - 2019) * 0.002
    solar_generation = np.where(
        day_mask,
        solar_irradiance * panel_efficiency * adjusted_prosumer_pct * city_cfg['household_count'] * 0.003,
        0.0,
    )

    grid_surplus = solar_generation - adjusted_demand

    district_code = ''.join([x[0] for x in district.split()]).upper()[:3]
    grid_zone_id = f'KA_{district_code}_{city_index:03d}'

    return pd.DataFrame({
        'timestamp': timestamps,
        'year': year,
        'month': month,
        'day': timestamps.day.values,
        'hour': hour,
        'day_of_week': dow,
        'is_weekend': is_weekend.astype(int),
        'is_holiday': is_holiday.astype(int),
        'season': season,
        'district': district,
        'city': city,
        'grid_zone_id': grid_zone_id,
        'zone_type': zone,
        'temperature_C': np.round(temperature, 2),
        'humidity_pct': np.round(humidity, 2),
        'solar_irradiance_Wm2': np.round(solar_irradiance, 2),
        'wind_speed_kmh': np.round(wind_speed, 2),
        'is_cloudy': is_cloudy.astype(int),
        'is_rainy': is_rainy.astype(int),
        'household_count': city_cfg['household_count'],
        'prosumer_pct': np.round(adjusted_prosumer_pct, 4),
        'base_demand_kWh': np.round(adjusted_base_demand, 4),
        'peak_flag': peak_flag,
        'weather_adjustment_factor': np.round(weather_adjustment_factor, 4),
        'adjusted_demand_kWh': np.round(adjusted_demand, 4),
        'panel_efficiency': np.round(panel_efficiency, 4),
        'solar_generation_kWh': np.round(solar_generation, 4),
        'grid_surplus_kWh': np.round(grid_surplus, 4),
    })

In [ ]:
all_frames = []

for year in range(2019, 2025):
    timestamps = pd.date_range(f'{year}-01-01 00:00:00', f'{year}-12-31 23:00:00', freq='H')

    for district, district_cfg in KARNATAKA_GRID.items():
        district_frames = []

        for idx, (city, city_cfg) in enumerate(district_cfg['cities'].items(), start=1):
            city_df = build_city_frame(district, city, city_cfg, year, timestamps, idx)
            district_frames.append(city_df)

        district_year_df = pd.concat(district_frames, ignore_index=True)
        district_slug = normalize_district_name(district)
        district_path = os.path.join(OUTPUT_DIR, f'{district_slug}_{year}.csv')
        district_year_df.to_csv(district_path, index=False)

        all_frames.append(district_year_df)

        print(f'Saved {district_path} -> shape={district_year_df.shape}')
        print(district_year_df.head(2))

combined_df = pd.concat(all_frames, ignore_index=True)
combined_path = os.path.join(OUTPUT_DIR, 'karnataka_energy_2019_2024.csv')
combined_df.to_csv(combined_path, index=False)

print('\nCombined dataset saved:', combined_path)
print('Combined shape:', combined_df.shape)
combined_df.head(5)